# Debug Notebook: Scheduler Rules Benchmark (Simulation)

This notebook benchmarks scheduler-rule optimization strategies for speculative decoding traffic:
- separate short vs long lanes
- avoid mixing tiny and long requests in same batch
- batch by similar remaining decode length
- continuous batching with a max queue-wait bound

It produces paired policy deltas, robust statistics (median/p90), and bootstrap CI outputs.

In [1]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except ModuleNotFoundError:
    torch = None

search_starts = []
if '__file__' in globals():
    search_starts.append(Path(__file__).resolve().parent)
search_starts.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_starts:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'speculative.py').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/speculative.py')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

os.environ.setdefault('SPECDEC_HF_OFFLINE_FIRST', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

SPECDEC_AVAILABLE = True
SPECDEC_IMPORT_ERROR = None
try:
    import speculative as spec_mod
    from config import DRAFT_MODELS, DRAFT_QUANT, TARGET_MODEL_ID, TARGET_QUANT, REGIMES
    from speculative import load_model_on_device, speculative_decode_sample
    from utils import set_seed
except ModuleNotFoundError as exc:
    if exc.name == 'transformers':
        SPECDEC_AVAILABLE = False
        SPECDEC_IMPORT_ERROR = exc
        spec_mod = None
        DRAFT_MODELS = {}
        DRAFT_QUANT = None
        TARGET_MODEL_ID = None
        TARGET_QUANT = None
        REGIMES = {}
    else:
        raise

results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
manifests_dir = project_root / 'manifests'

print(f'Project root: {project_root}')
print(f'CUDA available: {bool(torch is not None and torch.cuda.is_available())}')
if SPECDEC_AVAILABLE:
    print('Speculative runtime imports: available')
else:
    print(f'Speculative runtime imports unavailable: {SPECDEC_IMPORT_ERROR}')
    print('Will use existing profile CSV if present; new GPU profiling is disabled in this environment.')

Project root: C:\Working\speculative-decoding-main_v9\speculative-decoding-main
CUDA available: True
Speculative runtime imports: available


In [2]:
# Profiling + simulation controls
DEVICE = 'cuda:0'
DRAFT_LABEL = '0.5B'
REGIME_NAME = 'deterministic'
K = 8

PROFILE_MAX_NEW_TOKENS = [16, 64, 160]
N_PROMPTS_PER_TASK = 6
PROFILE_REPEATS = 1
USE_EXISTING_PROFILE_IF_AVAILABLE = True

# Safe profiling mode: disable speculative GPU micro-optimizations so scheduler
# profiling remains stable across driver/OS stacks (especially Windows CUDA).
SCHEDULER_PROFILE_SAFE_MODE = True

SHORT_MAX_TOKENS = 32
MEDIUM_MAX_TOKENS = 96

ARRIVAL_RATES_RPS = [0.20, 0.35, 0.50]
NUM_REQUESTS_PER_TRIAL = 300
NUM_TRIALS = 25
SIM_SEED = 2026

MAX_BATCH_SIZE = 8
BATCH_WINDOW_S = 0.040
MAX_QUEUE_WAIT_S = 0.060
SIMILARITY_RATIO_BOUND = 1.5

PROFILE_CSV = results_dir / 'gpu_scheduler_service_profile.csv'
RUNS_CSV = results_dir / 'gpu_scheduler_simulation_runs.csv'
DELTA_CSV = results_dir / 'gpu_scheduler_policy_delta.csv'
ROBUST_CSV = results_dir / 'gpu_scheduler_policy_robust_delta.csv'
BOOTSTRAP_CSV = results_dir / 'gpu_scheduler_policy_bootstrap_ci.csv'

print('Scheduler benchmark config ready.')
print(f'  safe_mode={SCHEDULER_PROFILE_SAFE_MODE}')

Scheduler benchmark config ready.
  safe_mode=True


In [3]:
def token_bucket(max_new_tokens: int) -> str:
    if max_new_tokens <= SHORT_MAX_TOKENS:
        return 'short'
    if max_new_tokens <= MEDIUM_MAX_TOKENS:
        return 'medium'
    return 'long'

def load_prompt_rows(n_per_task: int = 6) -> list[dict]:
    rows = []
    for task in ['gsm8k', 'mmlu', 'cnndm']:
        path = manifests_dir / f'{task}_data.json'
        if not path.exists():
            continue
        with open(path, 'r') as f:
            data = json.load(f)
        for item in data[:n_per_task]:
            prompt = item.get('prompt')
            if prompt:
                rows.append({
                    'task': task,
                    'sample_id': item.get('sample_id', ''),
                    'prompt': prompt,
                })
    if not rows:
        raise RuntimeError('No prompts found in manifests/*_data.json')
    return rows

prompt_rows = load_prompt_rows(N_PROMPTS_PER_TASK)
print(f'Loaded prompts: {len(prompt_rows)}')
pd.DataFrame(prompt_rows)[['task', 'sample_id']].head(10)

Loaded prompts: 18


,task,sample_id
0,gsm8k,gsm8k_2
1,gsm8k,gsm8k_11
2,gsm8k,gsm8k_15
3,gsm8k,gsm8k_21
4,gsm8k,gsm8k_22
5,gsm8k,gsm8k_32
6,mmlu,mmlu_abstract_algebra_0
7,mmlu,mmlu_abstract_algebra_1
8,mmlu,mmlu_abstract_algebra_2
9,mmlu,mmlu_abstract_algebra_3


In [4]:
def profile_service_times() -> pd.DataFrame:
    if USE_EXISTING_PROFILE_IF_AVAILABLE and PROFILE_CSV.exists():
        print(f'Using existing profile: {PROFILE_CSV}')
        return pd.read_csv(PROFILE_CSV)

    if not SPECDEC_AVAILABLE:
        raise RuntimeError(
            'Cannot generate a new service profile in this environment because transformers/speculative runtime is unavailable. '
            'Set USE_EXISTING_PROFILE_IF_AVAILABLE=True with an existing PROFILE_CSV, or run this notebook on a device with transformers installed.'
        )

    if torch is None or not torch.cuda.is_available():
        raise RuntimeError('CUDA is required to generate new service profile. Set USE_EXISTING_PROFILE_IF_AVAILABLE=True with an existing CSV.')

    if SCHEDULER_PROFILE_SAFE_MODE and spec_mod is not None:
        # Scheduler evaluation does not depend on these micro-optimization flags.
        # Disable them to avoid stream/cuda runtime incompatibilities in profiling.
        spec_mod.GPU_USE_SEPARATE_STREAMS = False
        spec_mod.GPU_PREALLOCATE_STEP_BUFFERS = False
        spec_mod.GPU_USE_STABLE_STEP_SHAPES = False
        spec_mod.GPU_TRY_CUDA_GRAPHS = False

    target_model, target_tokenizer = load_model_on_device(
        TARGET_MODEL_ID,
        device=DEVICE,
        quant_mode=TARGET_QUANT,
    )
    draft_model, _ = load_model_on_device(
        DRAFT_MODELS[DRAFT_LABEL],
        device=DEVICE,
        quant_mode=DRAFT_QUANT,
    )

    regime = REGIMES[REGIME_NAME]
    rows = []
    for rep in range(PROFILE_REPEATS):
        for i, p in enumerate(prompt_rows):
            for mnt in PROFILE_MAX_NEW_TOKENS:
                seed = SIM_SEED + rep * 100000 + i * 1000 + mnt
                set_seed(seed)

                try:
                    out = speculative_decode_sample(
                        target_model,
                        draft_model,
                        target_tokenizer,
                        p['prompt'],
                        max_new_tokens=mnt,
                        k=K,
                        temperature=regime.temperature,
                        top_p=regime.top_p,
                        return_timing_breakdown=True,
                    )
                except Exception as exc:
                    msg = str(exc)
                    hint = (
                        'Profiling failed during speculative_decode_sample.\n'
                        'If this is an Accelerator/CUDA runtime issue, keep '
                        'SCHEDULER_PROFILE_SAFE_MODE=True and/or provide an existing '
                        f'profile CSV at {PROFILE_CSV} with USE_EXISTING_PROFILE_IF_AVAILABLE=True.\n'
                        f'Original error: {msg}'
                    )
                    raise RuntimeError(hint) from exc

                rows.append({
                    'task': p['task'],
                    'sample_id': p['sample_id'],
                    'repeat': rep,
                    'max_new_tokens': mnt,
                    'bucket': token_bucket(mnt),
                    'latency_s': float(out.get('latency_s', 0.0)),
                    'draft_elapsed_s': float(out.get('draft_elapsed_s', 0.0)),
                    'verify_elapsed_s': float(out.get('verify_elapsed_s', 0.0)),
                    'ttft_ms': float(out.get('ttft_ms', 0.0)),
                    'num_tokens': int(out.get('num_tokens', 0)),
                })

    df = pd.DataFrame(rows)
    df.to_csv(PROFILE_CSV, index=False)
    print(f'Saved profile: {PROFILE_CSV}')
    return df

service_profile = profile_service_times()
service_profile.head()

Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Saved profile: C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\gpu_scheduler_service_profile.csv


,task,sample_id,repeat,max_new_tokens,bucket,latency_s,draft_elapsed_s,verify_elapsed_s,ttft_ms,num_tokens
0,gsm8k,gsm8k_2,0,16,short,0.8719,0.719331,0.151450,520.60,16
1,gsm8k,gsm8k_2,0,64,medium,2.9332,2.448165,0.481321,176.47,64
2,gsm8k,gsm8k_2,0,160,long,5.9945,4.994914,0.991335,181.69,160
3,gsm8k,gsm8k_11,0,16,short,0.3325,0.270473,0.061734,190.61,16
4,gsm8k,gsm8k_11,0,64,medium,1.3859,1.134658,0.249614,183.64,64


In [5]:
bucket_stats = service_profile.groupby('bucket', as_index=False).agg(
    mean_latency_s=('latency_s', 'mean'),
    p50_latency_s=('latency_s', 'median'),
    p90_latency_s=('latency_s', lambda x: float(np.quantile(x, 0.90))),
    n=('latency_s', 'count'),
)

latency_samples_by_bucket = {
    b: g['latency_s'].to_numpy()
    for b, g in service_profile.groupby('bucket')
}

if not latency_samples_by_bucket:
    raise RuntimeError('Service profile is empty.')

display(bucket_stats)

,bucket,mean_latency_s,p50_latency_s,p90_latency_s,n
0,long,6.304956,6.05420,7.74619,18
1,medium,2.566378,2.39730,3.56042,18
2,short,0.645356,0.59775,0.90316,18


In [6]:
POLICIES = [
    {
        'name': 'fifo_mixed_window',
        'split_lanes': False,
        'similarity_pick': False,
        'continuous': False,
    },
    {
        'name': 'lane_split_window',
        'split_lanes': True,
        'similarity_pick': False,
        'continuous': False,
    },
    {
        'name': 'length_similarity_window',
        'split_lanes': False,
        'similarity_pick': True,
        'continuous': False,
    },
    {
        'name': 'full_rules_continuous',
        'split_lanes': True,
        'similarity_pick': True,
        'continuous': True,
    },
]

POLICY_BASELINE = 'fifo_mixed_window'

def sample_bucket(rng: np.random.Generator) -> str:
    # Use empirical bucket mix from profile.
    counts = service_profile['bucket'].value_counts(normalize=True)
    buckets = counts.index.tolist()
    probs = counts.values
    return str(rng.choice(buckets, p=probs))

def sample_service_for_bucket(bucket: str, rng: np.random.Generator) -> float:
    arr = latency_samples_by_bucket[bucket]
    return float(rng.choice(arr))

def bucket_length_hint(bucket: str, rng: np.random.Generator) -> int:
    if bucket == 'short':
        return int(rng.integers(8, SHORT_MAX_TOKENS + 1))
    if bucket == 'medium':
        return int(rng.integers(SHORT_MAX_TOKENS + 1, MEDIUM_MAX_TOKENS + 1))
    return int(rng.integers(MEDIUM_MAX_TOKENS + 1, max(PROFILE_MAX_NEW_TOKENS) + 1))

def generate_requests(arrival_rate_rps: float, n: int, seed: int) -> list[dict]:
    rng = np.random.default_rng(seed)
    arrivals = np.cumsum(rng.exponential(scale=1.0 / arrival_rate_rps, size=n))
    reqs = []
    for i in range(n):
        bucket = sample_bucket(rng)
        reqs.append({
            'req_id': i,
            'arrival_s': float(arrivals[i]),
            'bucket': bucket,
            'remaining_tokens': bucket_length_hint(bucket, rng),
            'service_base_s': sample_service_for_bucket(bucket, rng),
        })
    return reqs

def effective_batch_service(batch: list[dict]) -> tuple[float, float]:
    # A simple throughput model: batching helps, heterogeneity hurts.
    base_services = np.array([r['service_base_s'] for r in batch], dtype=float)
    lengths = np.array([max(r['remaining_tokens'], 1) for r in batch], dtype=float)

    b = len(batch)
    size_speedup = min(1.40, 1.0 + 0.08 * (b - 1))
    hetero_penalty = 1.0 + 0.20 * float(np.std(lengths) / max(np.mean(lengths), 1e-9))
    batch_service = float(base_services.max() / size_speedup * hetero_penalty)
    return batch_service, float(hetero_penalty)

def pick_batch(pending: list[dict], cfg: dict) -> list[dict]:
    if not pending:
        return []

    if cfg['split_lanes']:
        short_lane = [r for r in pending if r['bucket'] in ('short', 'medium')]
        long_lane = [r for r in pending if r['bucket'] == 'long']
        if short_lane and long_lane:
            # Choose lane with older head request.
            use_short = short_lane[0]['arrival_s'] <= long_lane[0]['arrival_s']
            pending_view = short_lane if use_short else long_lane
        else:
            pending_view = short_lane if short_lane else long_lane
    else:
        pending_view = pending

    if cfg['similarity_pick'] and len(pending_view) > 1:
        anchor = pending_view[0]
        a_len = max(anchor['remaining_tokens'], 1)
        scored = []
        for r in pending_view:
            ratio = max(r['remaining_tokens'], 1) / a_len
            ratio = max(ratio, 1.0 / ratio)
            scored.append((ratio, abs(r['remaining_tokens'] - a_len), r))
        scored.sort(key=lambda x: (x[0], x[1], x[2]['arrival_s']))
        chosen = [t[2] for t in scored if t[0] <= SIMILARITY_RATIO_BOUND][:MAX_BATCH_SIZE]
        if not chosen:
            chosen = pending_view[:MAX_BATCH_SIZE]
        return chosen

    return pending_view[:MAX_BATCH_SIZE]

def simulate_policy(requests: list[dict], cfg: dict) -> pd.DataFrame:
    pending = []
    arrivals = sorted(requests, key=lambda r: r['arrival_s'])
    i = 0
    t = 0.0
    done = []

    while i < len(arrivals) or pending:
        if not pending and i < len(arrivals):
            t = max(t, arrivals[i]['arrival_s'])

        # Pull all arrivals up to current time.
        while i < len(arrivals) and arrivals[i]['arrival_s'] <= t:
            pending.append(arrivals[i])
            i += 1

        if not pending:
            continue

        pending.sort(key=lambda r: r['arrival_s'])
        oldest_wait = t - pending[0]['arrival_s']

        if cfg['continuous']:
            # In continuous mode, dispatch immediately when oldest request hits wait bound,
            # otherwise keep filling until next arrival or batch full.
            if len(pending) < MAX_BATCH_SIZE and oldest_wait < MAX_QUEUE_WAIT_S and i < len(arrivals):
                next_arrival = arrivals[i]['arrival_s']
                if next_arrival <= t + BATCH_WINDOW_S and next_arrival - pending[0]['arrival_s'] < MAX_QUEUE_WAIT_S:
                    t = next_arrival
                    continue
        else:
            # Windowed batching: wait a fixed window if possible before dispatch.
            if len(pending) < MAX_BATCH_SIZE and i < len(arrivals) and arrivals[i]['arrival_s'] <= t + BATCH_WINDOW_S:
                t = arrivals[i]['arrival_s']
                continue

        batch = pick_batch(pending, cfg)
        if not batch:
            t = arrivals[i]['arrival_s'] if i < len(arrivals) else t
            continue

        batch_ids = {r['req_id'] for r in batch}
        pending = [r for r in pending if r['req_id'] not in batch_ids]

        start_s = t
        batch_service_s, hetero_penalty = effective_batch_service(batch)
        finish_s = start_s + batch_service_s

        for r in batch:
            wait_s = max(0.0, start_s - r['arrival_s'])
            latency_s = finish_s - r['arrival_s']
            done.append({
                'req_id': r['req_id'],
                'bucket': r['bucket'],
                'arrival_s': r['arrival_s'],
                'remaining_tokens': r['remaining_tokens'],
                'service_base_s': r['service_base_s'],
                'wait_s': wait_s,
                'service_s': batch_service_s,
                'latency_s': latency_s,
                'batch_size': len(batch),
                'hetero_penalty': hetero_penalty,
            })

        t = finish_s

    return pd.DataFrame(done)

print('Simulation functions ready.')

Simulation functions ready.


In [7]:
sim_rows = []
for rate in ARRIVAL_RATES_RPS:
    for trial in range(NUM_TRIALS):
        req_seed = SIM_SEED + int(rate * 10000) + trial
        reqs = generate_requests(rate, NUM_REQUESTS_PER_TRIAL, req_seed)

        for cfg in POLICIES:
            per_req = simulate_policy(reqs, cfg)
            completion_s = per_req['arrival_s'] + per_req['latency_s']
            makespan_s = float(completion_s.max() - per_req['arrival_s'].min())
            safe_makespan_s = max(makespan_s, 1e-9)

            sim_rows.append({
                'arrival_rate_rps': rate,
                'trial': trial,
                'policy': cfg['name'],
                'n_requests': int(len(per_req)),
                'mean_latency_s': float(per_req['latency_s'].mean()),
                'p50_latency_s': float(np.quantile(per_req['latency_s'], 0.50)),
                'p90_latency_s': float(np.quantile(per_req['latency_s'], 0.90)),
                'mean_wait_s': float(per_req['wait_s'].mean()),
                'p90_wait_s': float(np.quantile(per_req['wait_s'], 0.90)),
                'mean_batch_size': float(per_req['batch_size'].mean()),
                'mean_hetero_penalty': float(per_req['hetero_penalty'].mean()),
                'throughput_rps': float(len(per_req) / safe_makespan_s),
            })

sim_runs = pd.DataFrame(sim_rows)
sim_runs.to_csv(RUNS_CSV, index=False)
print(f'Saved simulation runs: {RUNS_CSV}')
sim_runs.head()

Saved simulation runs: C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\gpu_scheduler_simulation_runs.csv


,arrival_rate_rps,trial,policy,n_requests,mean_latency_s,p50_latency_s,p90_latency_s,mean_wait_s,p90_wait_s,mean_batch_size,mean_hetero_penalty,throughput_rps
0,0.2,0,fifo_mixed_window,300,5.218209,5.236750,9.661401,1.474706,4.608918,1.500000,1.024474,0.196715
1,0.2,0,lane_split_window,300,5.254594,5.236750,9.749545,1.794459,5.700280,1.366667,1.015421,0.196715
2,0.2,0,length_similarity_window,300,5.334551,5.304668,9.958700,1.934903,5.737833,1.246667,1.004720,0.196715
3,0.2,0,full_rules_continuous,300,5.355483,5.304668,9.816363,2.006935,5.852756,1.213333,1.003790,0.196715
4,0.2,1,fifo_mixed_window,300,4.959143,4.851600,9.605713,1.300362,4.330114,1.453333,1.023775,0.199609


In [8]:
METRICS = ['mean_latency_s', 'p90_latency_s', 'mean_wait_s', 'p90_wait_s', 'throughput_rps']
KEYS = ['arrival_rate_rps', 'trial']

base = sim_runs[sim_runs['policy'] == POLICY_BASELINE][KEYS + METRICS].rename(columns={m: f'{m}_base' for m in METRICS})

paired_rows = []
for p in sorted(sim_runs['policy'].unique()):
    if p == POLICY_BASELINE:
        continue
    cmp_df = sim_runs[sim_runs['policy'] == p][KEYS + METRICS].rename(columns={m: f'{m}_cmp' for m in METRICS})
    merged = base.merge(cmp_df, on=KEYS, how='inner')
    if merged.empty:
        continue

    for m in METRICS:
        delta = merged[f'{m}_cmp'] - merged[f'{m}_base']
        merged[f'{m}_delta'] = delta
        merged[f'{m}_delta_pct'] = np.where(merged[f'{m}_base'] != 0, delta / merged[f'{m}_base'] * 100.0, np.nan)

    merged['baseline_policy'] = POLICY_BASELINE
    merged['compare_policy'] = p
    paired_rows.append(merged)

paired_df = pd.concat(paired_rows, ignore_index=True) if paired_rows else pd.DataFrame()

mean_rows = []
for p in sorted(paired_df['compare_policy'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_policy'] == p]
    for m in METRICS:
        b = float(block[f'{m}_base'].mean())
        c = float(block[f'{m}_cmp'].mean())
        d = float(block[f'{m}_delta'].mean())
        pct = (d / b * 100.0) if b != 0 else np.nan
        # For throughput, higher is better; invert sign for improvement metric.
        improve = pct if m == 'throughput_rps' else -pct
        mean_rows.append({
            'baseline_policy': POLICY_BASELINE,
            'compare_policy': p,
            'metric': m,
            'baseline_mean': b,
            'compare_mean': c,
            'abs_delta_compare_minus_base': d,
            'pct_delta_compare_minus_base': pct,
            'improvement_pct': improve,
            'n_pairs': int(len(block)),
        })

policy_delta = pd.DataFrame(mean_rows)
policy_delta.to_csv(DELTA_CSV, index=False)
display(policy_delta)

,baseline_policy,compare_policy,metric,baseline_mean,compare_mean,abs_delta_compare_minus_base,pct_delta_compare_minus_base,improvement_pct,n_pairs
0,fifo_mixed_window,full_rules_continuous,mean_latency_s,6.310580,7.086558,0.775978,12.296457,-12.296457,75
1,fifo_mixed_window,full_rules_continuous,p90_latency_s,10.930560,12.562610,1.632050,14.931070,-14.931070,75
2,fifo_mixed_window,full_rules_continuous,mean_wait_s,2.028766,3.825683,1.796917,88.571963,-88.571963,75
3,fifo_mixed_window,full_rules_continuous,p90_wait_s,5.154018,8.744747,3.590729,69.668547,-69.668547,75
4,fifo_mixed_window,full_rules_continuous,throughput_rps,0.348031,0.347093,-0.000937,-0.269315,-0.269315,75
5,fifo_mixed_window,lane_split_window,mean_latency_s,6.310580,6.419060,0.108480,1.719012,-1.719012,75
6,fifo_mixed_window,lane_split_window,p90_latency_s,10.930560,11.526177,0.595617,5.449097,-5.449097,75
7,fifo_mixed_window,lane_split_window,mean_wait_s,2.028766,2.920653,0.891887,43.962052,-43.962052,75
8,fifo_mixed_window,lane_split_window,p90_wait_s,5.154018,7.088824,1.934806,37.539770,-37.539770,75
9,fifo_mixed_window,lane_split_window,throughput_rps,0.348031,0.347354,-0.000677,-0.194539,-0.194539,75


In [9]:
robust_rows = []
for p in sorted(paired_df['compare_policy'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_policy'] == p]
    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        robust_rows.append({
            'baseline_policy': POLICY_BASELINE,
            'compare_policy': p,
            'metric': m,
            'median_delta': float(np.median(d)),
            'p90_delta': float(np.quantile(d, 0.90)),
            'p10_delta': float(np.quantile(d, 0.10)),
            'median_abs_delta': float(np.median(np.abs(d))),
            'p90_abs_delta': float(np.quantile(np.abs(d), 0.90)),
            'n_pairs': int(len(d)),
        })

robust_delta = pd.DataFrame(robust_rows)
robust_delta.to_csv(ROBUST_CSV, index=False)
display(robust_delta)

,baseline_policy,compare_policy,metric,median_delta,p90_delta,p10_delta,median_abs_delta,p90_abs_delta,n_pairs
0,fifo_mixed_window,full_rules_continuous,mean_latency_s,0.768388,1.468387,0.135991,0.768388,1.468387,75
1,fifo_mixed_window,full_rules_continuous,p90_latency_s,1.694329,3.080065,0.333805,1.694329,3.080065,75
2,fifo_mixed_window,full_rules_continuous,mean_wait_s,1.783756,2.855314,0.678798,1.783756,2.855314,75
3,fifo_mixed_window,full_rules_continuous,p90_wait_s,3.513608,5.604041,1.538056,3.513608,5.604041,75
4,fifo_mixed_window,full_rules_continuous,throughput_rps,-0.000253,0.000000,-0.003066,0.000322,0.003066,75
5,fifo_mixed_window,lane_split_window,mean_latency_s,0.088604,0.373765,-0.145301,0.144942,0.373765,75
6,fifo_mixed_window,lane_split_window,p90_latency_s,0.591427,1.254529,-0.193604,0.592606,1.254529,75
7,fifo_mixed_window,lane_split_window,mean_wait_s,0.890853,1.381631,0.348262,0.890853,1.381631,75
8,fifo_mixed_window,lane_split_window,p90_wait_s,1.981659,2.809081,0.923763,1.981659,2.809081,75
9,fifo_mixed_window,lane_split_window,throughput_rps,0.000000,0.000095,-0.002442,0.000212,0.002485,75


In [10]:
BOOTSTRAP_SAMPLES = 20000
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(BOOTSTRAP_SEED)

ci_rows = []
for p in sorted(paired_df['compare_policy'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_policy'] == p]
    n = len(block)
    if n == 0:
        continue
    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        means = np.empty(BOOTSTRAP_SAMPLES, dtype=float)
        for i in range(BOOTSTRAP_SAMPLES):
            idx = rng.integers(0, n, n)
            means[i] = float(d[idx].mean())
        lo, hi = np.quantile(means, [0.025, 0.975])
        ci_rows.append({
            'baseline_policy': POLICY_BASELINE,
            'compare_policy': p,
            'metric': m,
            'mean_delta': float(d.mean()),
            'ci95_low': float(lo),
            'ci95_high': float(hi),
            'ci_excludes_zero': bool((lo > 0) or (hi < 0)),
            'n_pairs': int(n),
            'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
        })

bootstrap_ci = pd.DataFrame(ci_rows)
bootstrap_ci.to_csv(BOOTSTRAP_CSV, index=False)
display(bootstrap_ci)

,baseline_policy,compare_policy,metric,mean_delta,ci95_low,ci95_high,ci_excludes_zero,n_pairs,bootstrap_samples
0,fifo_mixed_window,full_rules_continuous,mean_latency_s,0.775978,0.658674,0.895150,True,75,20000
1,fifo_mixed_window,full_rules_continuous,p90_latency_s,1.632050,1.404451,1.871075,True,75,20000
2,fifo_mixed_window,full_rules_continuous,mean_wait_s,1.796917,1.602900,1.993408,True,75,20000
3,fifo_mixed_window,full_rules_continuous,p90_wait_s,3.590729,3.249660,3.931690,True,75,20000
4,fifo_mixed_window,full_rules_continuous,throughput_rps,-0.000937,-0.001340,-0.000572,True,75,20000
5,fifo_mixed_window,lane_split_window,mean_latency_s,0.108480,0.063035,0.156461,True,75,20000
6,fifo_mixed_window,lane_split_window,p90_latency_s,0.595617,0.463262,0.729102,True,75,20000
7,fifo_mixed_window,lane_split_window,mean_wait_s,0.891887,0.802920,0.980484,True,75,20000
8,fifo_mixed_window,lane_split_window,p90_wait_s,1.934806,1.770511,2.098569,True,75,20000
9,fifo_mixed_window,lane_split_window,throughput_rps,-0.000677,-0.001037,-0.000349,True,75,20000


In [11]:
# Rate-wise summary and best policy pick by p90 latency.
rate_view = sim_runs.pivot_table(
    index='arrival_rate_rps',
    columns='policy',
    values='p90_latency_s',
    aggfunc='mean',
)
display(rate_view)

best_rows = []
for rate, g in sim_runs.groupby('arrival_rate_rps'):
    ranked = g.groupby('policy', as_index=False)['p90_latency_s'].mean().sort_values('p90_latency_s')
    best_rows.append({
        'arrival_rate_rps': float(rate),
        'best_policy_by_p90_latency': str(ranked.iloc[0]['policy']),
        'best_p90_latency_s': float(ranked.iloc[0]['p90_latency_s']),
    })

best_policy_table = pd.DataFrame(best_rows)
display(best_policy_table)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
sim_runs.to_csv(results_dir / f'gpu_scheduler_simulation_runs_{stamp}.csv', index=False)
policy_delta.to_csv(results_dir / f'gpu_scheduler_policy_delta_{stamp}.csv', index=False)
bootstrap_ci.to_csv(results_dir / f'gpu_scheduler_policy_bootstrap_ci_{stamp}.csv', index=False)

print('Saved scheduler simulation outputs:')
print(f'  {RUNS_CSV}')
print(f'  {DELTA_CSV}')
print(f'  {ROBUST_CSV}')
print(f'  {BOOTSTRAP_CSV}')

policy,fifo_mixed_window,full_rules_continuous,lane_split_window,length_similarity_window
arrival_rate_rps,,,,
0.20,10.093792,10.747050,10.322548,10.470543
0.35,11.160352,12.672125,11.705475,12.428775
0.50,11.537536,14.268653,12.550508,14.081442


,arrival_rate_rps,best_policy_by_p90_latency,best_p90_latency_s
0,0.20,fifo_mixed_window,10.093792
1,0.35,fifo_mixed_window,11.160352
2,0.50,fifo_mixed_window,11.537536


Saved scheduler simulation outputs:
  C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\gpu_scheduler_simulation_runs.csv
  C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\gpu_scheduler_policy_delta.csv
  C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\gpu_scheduler_policy_robust_delta.csv
  C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\gpu_scheduler_policy_bootstrap_ci.csv
